# Определение ориентации текста: 0° / 180°

## Финальное решение

**Leaderboard:** `0.94759798` (Brier = `0.05240202`)

Финальная схема:

1. `PP-LCNet_x1_0_textline_ori`
2. `PP-LCNet_x0_25_textline_ori`
3. TTA: дополнительный инференс на изображении, повернутом на 180°
4. Для каждой модели:
   `p_tta = (p_original + (1 - p_rotated)) / 2`
5. Объединение двух моделей в logit-пространстве:
   - x1.0: `0.59`
   - x0.25: `0.41`
6. Temperature scaling: `T = 0.36`

Решение работает на CPU и не использует внешние API или LLM/VLM для классификации.


## 1. Эксперименты и выбор финальной конфигурации

Параметры подбирались экспериментально до финальной отправки. Для выбора конфигурации использовалась отдельная синтетическая validation-выборка; labels тестовой выборки при этом не использовались.

На leaderboard были проверены следующие варианты:

| Конфигурация | Score `1 - Brier` |
|---|---:|
| x1.0 + x0.25 + TTA, `T=0.37` | 0.94756180 |
| x1.0 + x0.25 + TTA, `T=0.50` | 0.94671345 |
| x0.25-only, `T=0.37` | 0.92852895 |
| adaptive ensemble | 0.93900258 |
| **x1.0 + x0.25 + TTA, `T=0.36`** | **0.94759798** |

Таким образом, в финальную отправку вошла схема с весами `0.59 / 0.41` и температурой `0.36`.

Важно: эксперименты с synthetic validation использовались для разработки; данный notebook содержит минимальный воспроизводимый pipeline, который непосредственно генерирует финальный `submission.csv`.


## 2. Зависимости

Использованные версии окружения при получении финального результата:

- Python 3.12.7
- PaddlePaddle 3.2.0
- PaddleOCR 3.7.0
- NumPy 2.3.5
- Pandas 3.0.6
- SciPy 1.18.1
- Pillow

Модели являются готовыми моделями PaddleOCR для классификации ориентации текстовой строки:
`PP-LCNet_x1_0_textline_ori` и `PP-LCNet_x0_25_textline_ori`.

При первом запуске PaddleOCR загрузит официальные веса в локальный кэш.


In [1]:
%pip install -q paddlepaddle==3.2.0 paddleocr==3.7.0 pandas==3.0.6 numpy==2.3.5 pillow scipy==1.18.1


Note: you may need to restart the kernel to use updated packages.


## 3. Импорты и фиксированный seed

Seed фиксируется для воспроизводимости всех операций со случайностью в Python и NumPy. Сам финальный inference детерминирован при фиксированном окружении и тех же входных данных.


In [2]:
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from scipy.special import logit, expit
import paddle
import paddleocr

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Python:", sys.version.split()[0])
print("PaddlePaddle:", paddle.__version__)
print("PaddleOCR:", paddleocr.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Seed:", SEED)


C:\Users\Lecoo\Desktop\avito\.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Python: 3.12.7
PaddlePaddle: 3.2.0
PaddleOCR: 3.7.0
NumPy: 2.3.5
Pandas: 3.0.6
Seed: 42


## 4. Пути к данным

Notebook предполагает запуск из корня проекта, где находятся выданные тестовые данные.

Ожидаемая структура:

```text
project/
├── test/
│   └── test/
│       └── images/
└── .../sample_submission.csv
```

Повороты на 180° сохраняются в отдельную папку `test/rot180`, поэтому при повторном запуске они не создаются заново.


In [3]:
PROJECT_DIR = Path.cwd()

TEST_IMAGES_DIR = PROJECT_DIR / "test" / "test" / "images"
ROT_DIR = PROJECT_DIR / "test" / "rot180"
SUBMISSION_PATH = PROJECT_DIR / "submission.csv"

sample_candidates = sorted(
    PROJECT_DIR.rglob("sample_submission.csv")
)

if not sample_candidates:
    raise FileNotFoundError(
        "Не найден sample_submission.csv. Проверьте PROJECT_DIR."
    )

SAMPLE_SUBMISSION = sample_candidates[0]

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg"}

images = sorted(
    p for p in TEST_IMAGES_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

print("PROJECT_DIR:", PROJECT_DIR)
print("TEST_IMAGES_DIR:", TEST_IMAGES_DIR)
print("SAMPLE_SUBMISSION:", SAMPLE_SUBMISSION)
print("Test images:", len(images))

assert len(images) == 20000, (
    f"Ожидалось 20000 изображений, найдено {len(images)}"
)


PROJECT_DIR: C:\Users\Lecoo\Desktop\avito
TEST_IMAGES_DIR: C:\Users\Lecoo\Desktop\avito\test\test\images
SAMPLE_SUBMISSION: C:\Users\Lecoo\Desktop\avito\proverka\test\sample_submission.csv
Test images: 20000


## 5. TTA: поворот изображения на 180°

Для каждого тестового изображения создаётся точная копия, повернутая на 180°.

Если исходная вероятность класса `180°` равна `p_original`, а для повернутого изображения модель возвращает `p_rotated`, то после поворота смысл классов меняется местами. Поэтому вторую оценку приводим к исходной системе координат как `1 - p_rotated`.

Это позволяет получить более устойчивую оценку ориентации.


In [4]:
ROT_DIR.mkdir(parents=True, exist_ok=True)

for image_path in images:
    rotated_path = ROT_DIR / image_path.name

    if not rotated_path.exists():
        with Image.open(image_path) as img:
            img.convert("RGB").transpose(
                Image.Transpose.ROTATE_180
            ).save(rotated_path)

rot_images = sorted(
    p for p in ROT_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

print("Original images:", len(images))
print("Rotated images:", len(rot_images))

assert len(rot_images) == 20000


Original images: 20000
Rotated images: 20000


## 6. Загрузка моделей

Используются две готовые модели PaddleOCR:

- `PP-LCNet_x1_0_textline_ori` — более крупная модель;
- `PP-LCNet_x0_25_textline_ori` — компактная модель.

Обе модели решают именно задачу классификации ориентации текстовой строки на классы `0_degree` и `180_degree`.


In [5]:
from paddleocr import TextLineOrientationClassification

model_x10 = TextLineOrientationClassification(
    model_name="PP-LCNet_x1_0_textline_ori",
    device="cpu"
)

model_x025 = TextLineOrientationClassification(
    model_name="PP-LCNet_x0_25_textline_ori",
    device="cpu"
)

print("Models initialized successfully.")


Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Lecoo\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Lecoo\.paddlex\official_models\PP-LCNet_x0_25_textline_ori`.


Models initialized successfully.


## 7. Получение вероятности `p_180`

PaddleOCR возвращает предсказанный класс и его score. Если предсказанный класс — `180_degree`, score непосредственно является `p_180`. Если предсказан `0_degree`, вероятность `180°` равна `1 - score`.


In [6]:
def predict_p180(model, paths, batch_size=32):
    paths = [str(p) for p in paths]
    rows = []

    for res in model.predict_iter(
        input=paths,
        batch_size=batch_size
    ):
        data = res.json["res"]

        score = float(data["scores"][0])
        label = data["label_names"][0]

        p180 = (
            score
            if label == "180_degree"
            else 1.0 - score
        )

        rows.append({
            "image_id": Path(data["input_path"]).stem,
            "p_180": p180
        })

    result = pd.DataFrame(rows)

    if result.empty:
        raise RuntimeError(
            "PaddleOCR не вернул результатов. "
            "Проверьте входные данные и окружение."
        )

    return result


## 8. Четыре inference-прохода

Для каждой из двух моделей выполняется inference:

1. на исходных изображениях;
2. на изображениях, повернутых на 180°.

Итого получаем четыре оценки на каждую тестовую картинку.


In [7]:
x10_orig = predict_p180(
    model_x10, images, batch_size=32
).rename(columns={"p_180": "p_x10_orig"})

x10_rot = predict_p180(
    model_x10, rot_images, batch_size=32
).rename(columns={"p_180": "p_x10_rot"})

x025_orig = predict_p180(
    model_x025, images, batch_size=32
).rename(columns={"p_180": "p_x025_orig"})

x025_rot = predict_p180(
    model_x025, rot_images, batch_size=32
).rename(columns={"p_180": "p_x025_rot"})

print("x10_orig :", x10_orig.shape)
print("x10_rot  :", x10_rot.shape)
print("x025_orig:", x025_orig.shape)
print("x025_rot :", x025_rot.shape)

assert all(
    df.shape[0] == 20000
    for df in [x10_orig, x10_rot, x025_orig, x025_rot]
)


x10_orig : (20000, 2)
x10_rot  : (20000, 2)
x025_orig: (20000, 2)
x025_rot : (20000, 2)


## 9. TTA + logit ensemble + temperature scaling

Для каждой модели вычисляется TTA-оценка:

```text
p_tta = (p_original + 1 - p_rotated) / 2
```

Затем две модели объединяются в logit-пространстве. Это позволяет комбинировать уверенность моделей не простым арифметическим средним.

Финальные параметры:

- `W_X10 = 0.59`
- `W_X025 = 0.41`
- `T = 0.36`

Температура `0.36` — единственное изменение относительно предыдущей проверенной конфигурации `T=0.37`, которое дало новый лучший leaderboard score.


In [8]:
test_pred = (
    x10_orig
    .merge(x10_rot, on="image_id", how="inner")
    .merge(x025_orig, on="image_id", how="inner")
    .merge(x025_rot, on="image_id", how="inner")
)

assert test_pred.shape == (20000, 5)

test_pred["p_tta_x10"] = (
    test_pred["p_x10_orig"]
    + (1.0 - test_pred["p_x10_rot"])
) / 2.0

test_pred["p_tta_x025"] = (
    test_pred["p_x025_orig"]
    + (1.0 - test_pred["p_x025_rot"])
) / 2.0

W_X10 = 0.59
W_X025 = 0.41
TEMPERATURE = 0.36

p_x10 = np.clip(
    test_pred["p_tta_x10"].to_numpy(dtype=float),
    1e-7, 1.0 - 1e-7
)

p_x025 = np.clip(
    test_pred["p_tta_x025"].to_numpy(dtype=float),
    1e-7, 1.0 - 1e-7
)

z_ensemble = (
    W_X10 * logit(p_x10)
    + W_X025 * logit(p_x025)
)

test_pred["p_final"] = expit(
    z_ensemble / TEMPERATURE
)

print(test_pred["p_final"].describe())


count    2.000000e+04
mean     5.018757e-01
std      4.574388e-01
min      3.593815e-20
25%      3.078866e-04
50%      5.003479e-01
75%      9.997862e-01
max      1.000000e+00
Name: p_final, dtype: float64


## 10. Формирование финального `submission.csv`

Порядок `image_id` берётся из `sample_submission.csv`, а не из порядка выполнения inference.

Перед сохранением выполняются проверки:

- ровно 20 000 строк;
- ровно две колонки `image_id`, `p_180`;
- отсутствуют пропуски;
- все вероятности находятся в `[0, 1]`;
- порядок `image_id` полностью совпадает с sample submission.


In [9]:
sample = pd.read_csv(SAMPLE_SUBMISSION)

submission = (
    sample[["image_id"]]
    .merge(
        test_pred[["image_id", "p_final"]].rename(
            columns={"p_final": "p_180"}
        ),
        on="image_id",
        how="left",
        validate="one_to_one"
    )
)

assert submission.shape == (20000, 2)
assert submission.columns.tolist() == ["image_id", "p_180"]
assert submission["p_180"].notna().all()
assert submission["image_id"].equals(sample["image_id"])
assert submission["p_180"].between(0, 1).all()

submission.to_csv(
    SUBMISSION_PATH,
    index=False
)

print("=" * 60)
print("ALL CHECKS PASSED")
print("=" * 60)
print("Saved:", SUBMISSION_PATH)
print("Rows:", len(submission))
print("NaN:", submission["p_180"].isna().sum())
print("Mean p_180:", submission["p_180"].mean())
print()
print(submission.head())


ALL CHECKS PASSED
Saved: C:\Users\Lecoo\Desktop\avito\submission.csv
Rows: 20000
NaN: 0
Mean p_180: 0.5018757285841641

     image_id     p_180
0  test_00000  0.135844
1  test_00001  0.999718
2  test_00002  1.000000
3  test_00003  0.018978
4  test_00004  0.999925


## 11. Воспроизводимость и ограничения

- Все используемые случайные seeds зафиксированы (`seed = 42`).
- Финальный pipeline не использует labels тестовой выборки.
- Инференс выполняется на CPU.
- Внешние API и LLM/VLM для классификации не используются.
- При первом запуске PaddleOCR должен иметь возможность скачать официальные веса моделей; после этого они хранятся в локальном кэше.
- Для полного воспроизведения нужен тот же набор входных изображений и `sample_submission.csv`.

**Результат этого notebook:** `submission.csv`.
